In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ['USER_AGENT'] = 'Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0'
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
from langchain_classic import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
langchain.__version__


'1.5.0'

In [2]:
with open("data/secrets/key.secret") as f:
    os.environ["OPENAI_API_KEY"] = f.read()
with open("data/secrets/tavily.secret") as f:
    os.environ["TAVILY_API_KEY"] = f.read()

In [3]:
tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)
tool.invoke({"query": "Quel météo fait il à Grenoble ?"})

{'query': 'Quel météo fait il à Grenoble ?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://meteofrance.com/previsions-meteo-france/grenoble/38000',
   'title': 'METEO GRENOBLE par Météo-France - Prévisions Météo gratuites pour aujourd’hui, demain et à 15 jours.',
   'content': "Pour samedi après-midi. Soleil, puis les nuages deviennent plus nombreux à la tombée du jour.  Température : 11 degrés vers 13 heures.  Vent faible.    Pour dimanche matin. Temps changeant donnant des averses en cours de matinée ; bien ensoleillé ensuite.  Cumul de précipitations : 1 millimètres.  Températures minimales : 7 degrés.  Vent faible de direction variable.    Pour dimanche après-midi. Temps voilé, puis dégradation nuageuse à la tombée du jour.  Températures maximales : 15 degrés. [...] : 15 degrés.  Vent faible.    Pour lundi matin. Temps changeant accompagné d'ondées éparses en matinée.  La quantité de précipitations prévue est de l'ordre de 3 millimètres

In [6]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
agent = create_react_agent(llm, [tool])

user_input = "What nation hosted the Euro 2024? Include only wikipedia sources."

for step in agent.stream(
    {"messages": user_input},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What nation hosted the Euro 2024? Include only wikipedia sources.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_pXNTuFIsnK5vYpZLX9GRJbN4)
 Call ID: call_pXNTuFIsnK5vYpZLX9GRJbN4
  Args:
    query: Euro 2024 host nation
    include_domains: ['wikipedia.org']
================================= Tool Message =================================
Name: tavily_search

{"query": "Euro 2024 host nation", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "http://en.wikipedia.org/wiki/UEFA_Euro_2024", "title": "UEFA Euro 2024 - Wikipedia", "content": "| Fußball-Europameisterschaft 2024 (in German) |\n| United by Football. Vereint im Herzen Europas. (United in the heart of Europe.) |\n| Tournament details |\n| Host country | Germany |\n| Dates | 14 June – 14 July |\n| Teams | 24 |\n| Venues | 10 (in 10 host cities) |\n| Fina